# Chapter 7 — The Dashboard

**Time:** ~45 minutes  
**Goal:** Understand Streamlit, run the dashboard, and map every UI element to its code.

---

## 7.1 What is Streamlit?

Streamlit is a Python library that turns Python scripts into interactive web applications — no HTML, CSS, or JavaScript required.

You write Python. Streamlit handles the web page.

Install it:
```bash
pip install streamlit
```

Run a Streamlit app:
```bash
streamlit run src/dashboard.py
```

Your browser opens automatically at `http://localhost:8501`.

## 7.2 The 8 Streamlit Functions Used in This Project

You do not need to know all of Streamlit. You only need the functions actually used in `dashboard.py`.

| Function | What it shows |
|---|---|
| `st.title(text)` | Large heading at the top of the page |
| `st.caption(text)` | Small grey subtitle text |
| `st.metric(label, value)` | A bold number with a label — used for Temperature, Humidity, Energy |
| `st.error(text)` | A red alert box |
| `st.warning(text)` | An orange warning box |
| `st.success(text)` | A green box |
| `st.info(text)` | A blue info box |
| `st.line_chart(dataframe)` | A line chart from a pandas DataFrame |
| `st.dataframe(data)` | A table |
| `st.columns(n)` | Splits the page into n side-by-side columns |
| `st.sidebar.slider(...)` | A slider control in the left sidebar |
| `st.rerun()` | Reruns the entire script (creates the refresh loop) |

## 7.3 A Minimal Streamlit App

Create a new file `hello_streamlit.py` anywhere and paste this:

```python
import streamlit as st

st.title("My First Streamlit App")
st.metric("Temperature", "29.5°C")
st.metric("Humidity", "68.2%")

if st.button("Click me"):
    st.success("Button clicked!")
```

Run it:
```bash
streamlit run hello_streamlit.py
```

You will see a page with a title, two metrics, and a clickable button. Each `st.` call adds one element to the page, top to bottom.

## 7.4 How the Dashboard Auto-Refreshes

The dashboard updates itself every 2 seconds. Here is how:

```python
def main():
    ...              # 1. Run one simulation cycle
    ...              # 2. Render everything
    time.sleep(2)    # 3. Wait 2 seconds
    st.rerun()       # 4. Start main() again from the top
```

`st.rerun()` restarts the entire script. This creates an infinite loop: render → wait → render → wait → ...

Because the script restarts from scratch each time, Streamlit needs a way to keep the simulation objects alive between runs. This is what `st.session_state` does.

## 7.5 st.session_state — Keeping Objects Alive

Normally, every variable in a Python script is destroyed when the script ends. Since `st.rerun()` restarts the script, all your objects would reset to zero every 2 seconds.

`st.session_state` is a special dictionary that **persists across reruns**. Anything you store there survives the restart.

```python
def init_session():
    if "simulator" not in st.session_state:
        st.session_state.simulator  = SensorSimulator()
        st.session_state.engine     = DecisionEngine()
        st.session_state.calculator = EnergyCalculator()
        st.session_state.logger     = DataLogger()
```

The `if "simulator" not in st.session_state` check means: only create new objects on the **first** run. On every subsequent run, the existing objects are reused — so cumulative energy keeps growing, the logger keeps accumulating records, and the sensor keeps its `last_temp`.

## 7.6 Reading dashboard.py — Function by Function

Open `src/dashboard.py`. The file has 5 functions:

| Function | Job |
|---|---|
| `init_session()` | Creates simulation objects on first run |
| `render_sidebar()` | Draws sliders and controls; returns current parameter values |
| `render_decision_reason(reason)` | Shows a coloured box explaining the latest AC decision |
| `render_dashboard(records, params)` | Draws everything: metrics, charts, table |
| `run_cycle(params)` | Runs one simulation step and stores the record |
| `main()` | Calls all the above in order, then sleeps and reruns |

**The flow inside `main()`:**
```python
def main():
    st.set_page_config(...)     # page title and layout
    st.title(...)               # show the big heading

    init_session()              # ensure objects exist
    params = render_sidebar()   # draw sidebar, get slider values
    run_cycle(params)           # one simulation step

    records = st.session_state.logger.get_latest(50)
    render_dashboard(records, params)  # draw everything

    time.sleep(2)
    st.rerun()
```

## 7.7 What Each UI Element Corresponds to in Code

Run the dashboard and identify each element:

| What you see on screen | Code in dashboard.py |
|---|---|
| "⚡ Smart Energy Optimization System" heading | `st.title(...)` in `main()` |
| Temperature Threshold slider (sidebar) | `st.sidebar.slider("Temperature Threshold...")` in `render_sidebar()` |
| "AC turns ON above X°C" text (sidebar) | `st.sidebar.markdown(...)` in `render_sidebar()` |
| Coloured decision box | `render_decision_reason(latest["reason"])` |
| Temperature metric | `col1.metric("Temperature (°C)", ...)` in `render_dashboard()` |
| Green/red AC status box | `col4.success(...)` or `col4.error(...)` |
| Temperature line chart | `st.line_chart(temp_df)` |
| Recent log table | `st.dataframe(table, ...)` |
| High temperature alert | `st.error("⚠️ High Temperature Alert...")` |

## 7.8 What pandas Does Here

You may notice `import pandas as pd` at the top. pandas is used in only one place — to create DataFrames for the two charts:

```python
temp_df = pd.DataFrame({
    "Temperature (°C)": [r["temperature"] for r in records],
    "AC ON above (27.5°C)": [27.5 for _ in records],
    "AC OFF below (24.5°C)": [24.5 for _ in records],
})
st.line_chart(temp_df)
```

A `pd.DataFrame` is just a table — rows and columns, like a spreadsheet. `st.line_chart()` expects a DataFrame. Each column in the DataFrame becomes one line on the chart.

The `[r["temperature"] for r in records]` part is a **list comprehension** — a compact way to build a list from another list:

In [ ]:
records = [
    {"temperature": 28.5, "cycle_energy": 2.0},
    {"temperature": 27.1, "cycle_energy": 2.0},
    {"temperature": 25.3, "cycle_energy": 0.1},
]

# List comprehension — build a list by extracting one field from each record
temperatures = [r["temperature"] for r in records]
print(temperatures)   # [28.5, 27.1, 25.3]

# Equivalent with a regular for loop:
temperatures2 = []
for r in records:
    temperatures2.append(r["temperature"])
print(temperatures2)  # same result

---
## Exercises

**Exercise 1:** Run the dashboard (`streamlit run src/dashboard.py`). Move the Temperature Threshold slider. Describe what changes on the page and explain why.

*Your answer here:*

**Exercise 2:** In `render_dashboard()`, find the line that shows the AC status as green or red. What is the condition that makes it green?

*Your answer here:*

**Exercise 3:** Write a list comprehension that extracts `cycle_energy` from every record in the list below:

In [ ]:
records = [
    {"temperature": 29.0, "cycle_energy": 2.0},
    {"temperature": 26.5, "cycle_energy": 0.1},
    {"temperature": 28.0, "cycle_energy": 2.0},
    {"temperature": 24.0, "cycle_energy": 0.1},
]

# Write your list comprehension here
energies = []
print(energies)

---
**Chapter 7 complete.** Move on to Chapter 8 — Experiments.